# 🏗️ Notebook 1: S3 (Object Storage) — Requirements & Architecture

Welcome! In this lab we're going to design a simplified version of
**Amazon S3** — a service that stores *objects* (arbitrary blobs of
bytes) behind a simple HTTP API.

By the end of this notebook you should be able to answer:

- What is object storage, and how is it different from a file system?
- Why is durability such a big deal, and how do we even *measure* it?
- What does the 10,000-foot architecture look like?

The next two notebooks zoom in on the **data model & APIs** and on a
few **deep-dive algorithms** (presigned URLs, consistent hashing,
erasure coding, multipart uploads).


## 🛠️ Setup

```bash
cd 06-system-designs/s3
uv sync
```

Then in VS Code pick the `.venv` kernel from the top-right of the notebook. If
it doesn't show up: `Cmd+Shift+P` → **Reload Window** and try again.

Everything in this lab is **pure Python** — no databases, no Docker. You can
run it on a laptop in a few seconds.


## 🧠 What *is* object storage?

Three ways people store data on a computer:

| Kind | Analogy | Access pattern |
|---|---|---|
| **Block storage** (e.g. EBS, a hard drive) | A raw notebook of empty pages | Read/write specific offsets — the OS decides what goes where |
| **File storage** (e.g. NFS, your laptop's disk) | A filing cabinet with folders and files | Open/read/write/seek, rename, permissions |
| **Object storage** (e.g. S3) | A giant parking lot where each car has a printed ticket | `PUT`/`GET`/`DELETE` a whole blob, addressed by a key |

In object storage, **the smallest unit you can touch is the whole
object**. You can't open `cat.jpg` and patch byte 42 — you replace the
entire thing, and you get a new version. That one rule is what makes
the system scale to exabytes.


## 🎯 Requirements

### Functional
- Create & delete **buckets** (a bucket is just a namespace).
- `PUT`, `GET`, `LIST`, `DELETE` objects inside a bucket.
- **Versioning** per bucket so an accidental overwrite isn't fatal.
- **Multipart upload** for large objects (multi-GB files).
- **Presigned URLs** — short-lived links that grant one specific
  action without handing out your credentials.
- **Cross-region replication** for disaster recovery.

### Non-functional
- **Durability ≈ 11 nines** (99.999999999%). One in 10¹¹ objects
  might be lost in a year. That is the headline number.
- **Availability ≈ 4 nines** (99.99%). A bit of downtime is fine; a
  lost object is not.
- **Scale**: petabytes today, exabytes later. ~100 M new objects/day
  (we compute what that means in PUT/s below rather than guessing).
- **Cheap**: cost per GB-month matters more than microsecond latency.
- **Consistency**: strong read-after-write for a single key is nice,
  but listing and replication can be eventually consistent.

## ✏️ Back-of-envelope

Before we draw any boxes, let's play with numbers. We'll assume a
modest workload and see where the first bottleneck hits.


In [ ]:
# Back-of-envelope estimate
daily_uploads  = 100_000_000        # 100 M new objects/day (a modest workload)
avg_size_bytes = 1 * 1024 * 1024    # 1 MB average object
read_write_ratio = 10               # GETs per PUT — object stores are read-heavy

bytes_per_day = daily_uploads * avg_size_bytes
pb_per_day = bytes_per_day / (1024 ** 5)
print(f"~{pb_per_day:.2f} PB of new data per day")

# Peak QPS. Note the assumption: essentially all of the day's traffic lands in a
# 6-hour window. Say that out loud — it is doing a lot of work in this number.
peak_window_seconds = 6 * 3600
peak_put_qps = daily_uploads / peak_window_seconds
print(f"~{peak_put_qps:,.0f} PUT/s and ~{peak_put_qps*read_write_ratio:,.0f} GET/s at peak")
print(f"  (not 'millions of PUT/s' — always derive the QPS, never assert it)")

# 10-year retention
ten_year_pb = pb_per_day * 365 * 10
print(f"~{ten_year_pb:,.0f} PB = {ten_year_pb/1024:.2f} EB over 10 years (ignoring deletes)")
print()

# --- Bandwidth, and the amplification the data plane actually sees --------
EC_K, EC_M = 10, 4
ec_overhead = (EC_K + EC_M) / EC_K
ingress_gbps = bytes_per_day / peak_window_seconds * 8 / 1e9
print(f"Client ingress at peak     : {ingress_gbps:6.1f} Gbps")
print(f"Disk writes at peak (10+4) : {ingress_gbps*ec_overhead:6.1f} Gbps  (x{ec_overhead:.1f} for parity)")
print(f"Client egress at peak      : {ingress_gbps*read_write_ratio:6.1f} Gbps")
print()

# --- The metadata plane: small in bytes, huge in row count ----------------
# This is the estimate people skip, and it is the one that decides the design.
meta_bytes_per_object = 400          # key, version, size, etag, shard map, ACL, timestamps
ten_year_objects = daily_uploads * 365 * 10
meta_tb = ten_year_objects * meta_bytes_per_object / 1e12
print(f"Objects after 10 years : {ten_year_objects/1e9:,.0f} B")
print(f"Metadata volume        : {meta_tb:,.0f} TB "
      f"({meta_tb*1e12/bytes_per_day/365/10*100:.4f}% the size of the data)")
print(f"Metadata QPS at peak   : {peak_put_qps*(1+read_write_ratio):,.0f}/s "
      f"(every GET and PUT is a metadata lookup first)")
print()
print("→ 365 billion rows is far past one database, so the metadata plane is")
print("  sharded — and *how* you shard it decides whether LIST is cheap or")
print("  catastrophic. That is the deep dive in Notebook 3.")

Two takeaways, and one correction to the intuition people usually bring here.

- **The data plane is a cost problem.** 0.33 EB over ten years is not "an
  exabyte" — be precise, the difference is a nine-figure hardware bill — but it
  is still far past anything you keep three copies of. Erasure coding, next.
- **The metadata plane is a *row count* problem.** 365 billion rows holding
  ~146 TB. Trivially small next to the data, utterly impossible for one
  database. Every GET and PUT hits it first, so it also carries **11× the QPS**
  of the data plane.
- **The correction:** newcomers assume object storage is hard because the
  objects are big. It isn't. Big blobs are the easy part — they are immutable,
  written once, read sequentially. The hard parts are the index over them and
  the durability guarantee. Both live in the metadata plane.

## 💸 Durability vs cost: replication vs erasure coding

The obvious way to avoid losing data is to keep N copies. The problem
is cost: 3 copies = 3× the hardware bill.

**Erasure coding** is the clever alternative. We split each object
into `k` data shards and compute `m` parity shards. Any `k` of the
`k+m` shards are enough to rebuild the object.

Example: 10+4 means we keep 14 shards total and can lose any 4 of them.


In [ ]:
# Storage overhead: replication vs erasure coding
def overhead_replication(copies: int) -> float:
    return copies

def overhead_erasure(k: int, m: int) -> float:
    return (k + m) / k

schemes = [
    ("1x (single copy, no redundancy)", overhead_replication(1)),
    ("3x replication",                  overhead_replication(3)),
    ("Erasure 10+4",                    overhead_erasure(10, 4)),
    ("Erasure 17+3",                    overhead_erasure(17, 3)),
]

raw_pb = 1000  # 1 EB of logical data
print(f"{'scheme':<35} {'overhead':>8}  {'physical PB for 1 EB':>22}")
for name, ov in schemes:
    print(f"{name:<35} {ov:>7.2f}x  {raw_pb*ov:>22,.0f}")


Going from 3× replication to 10+4 erasure coding cuts physical storage by
**more than half**. At exabyte scale that is billions of dollars. This is why
every serious object store uses erasure coding for its main tier.

**And here is what it costs you**, which the storage-overhead table hides:

| Cost | Why | Who feels it |
|---|---|---|
| **Read amplification on repair** | Rebuilding one lost shard requires reading **k** shards (10, for 10+4) and decoding. Replication just copies one. | The network, during exactly the incident you are trying to recover from |
| **Every read touches k nodes** | A replicated read hits one disk; an EC read gathers 10 fragments, so your latency is the **slowest of ten** — a classic tail-latency amplifier | p99 GET latency |
| **CPU on write and on every degraded read** | Reed–Solomon over GF(2⁸). Cheap per GB, not free at exabyte throughput | Cost per PUT |
| **Terrible for small objects** | A 4 KB object split 10 ways gives 400-byte fragments; per-fragment metadata and disk-sector overhead can exceed the data | Anyone storing lots of tiny objects |
| **Slower small-range reads** | Reading 1 KB out of the middle may require reconstructing a whole stripe | Analytics workloads |

Real systems therefore run **both**: replication for small and hot objects,
erasure coding for large and cold ones, with a background job that re-encodes as
objects cool. "We use erasure coding" is only half an answer — the full answer
names the crossover size.

We'll implement a tiny erasure code in Notebook 3.

## 🎲 Where does "eleven nines" *actually* come from?

Durability is a probability, not a promise. The standard back-of-envelope goes:
assume each shard fails independently with annual probability `p`; we lose the
object only if more than `m` of the `k+m` shards are gone, because any `k`
survivors can rebuild it.

Run that model and watch it fail to produce eleven nines. Then we'll fix it.

In [ ]:
from math import comb, log10

AFR   = 0.04       # annual failure rate of one disk — 4% is typical for a fleet
HOURS = 8760

def nines(p):
    return float("inf") if p <= 0 else -log10(p)

# ---------- Model A (the one everybody writes first): a whole year ---------
def loss_prob_year(k, m, afr=AFR):
    """P(more than m of the k+m shards fail *at some point during the year*)."""
    n = k + m
    return sum(comb(n, f) * afr**f * (1 - afr)**(n - f) for f in range(m + 1, n + 1))

# ---------- Model B: what actually matters — the REPAIR window ------------
def loss_prob_repair(k, m, afr=AFR, repair_hours=4.0):
    """Losing an object needs m+1 failures *before the first one is repaired*.

    A shard that failed in January and was rebuilt in January cannot conspire
    with a shard that fails in July. Model A quietly assumes it can, which is
    why it is wildly pessimistic.

      p_w = probability a given shard dies inside one repair window
      rate = n * afr             (first failures per object-year)
      each is fatal only if m more of the remaining n-1 shards die in the window
    """
    n = k + m
    p_w = afr * repair_hours / HOURS
    return n * afr * comb(n - 1, m) * p_w**m

schemes = [("3x replication", 1, 2), ("EC 6+3", 6, 3), ("EC 10+4", 10, 4), ("EC 17+3", 17, 3)]

print(f"{'scheme':<16}{'overhead':>9}{'Model A: whole year':>22}{'Model B: 4h repair':>22}")
for name, k, m in schemes:
    a, b = loss_prob_year(k, m), loss_prob_repair(k, m)
    print(f"{name:<16}{(k+m)/k:>8.2f}x{a:>13.1e} ({nines(a):>4.1f} 9s){b:>13.1e} ({nines(b):>4.1f} 9s)")

print()
worst = loss_prob_year(17, 3)
print(f"Model A says even 10+4 is only {nines(loss_prob_year(10,4)):.1f} nines, and 17+3 is {nines(worst):.1f} —")
print(f"i.e. it predicts losing 1 object in {1/worst:,.0f} every year. That is absurd; no")
print("storage vendor would survive it. The model is wrong, not the hardware.")
print()
print("What Model A gets wrong: it treats a shard that died in January and was")
print("rebuilt the same day as still 'failed' in July, so it can gang up with a")
print("July failure. Real systems repair continuously — the object is only at")
print("risk during the window between a failure and its repair.")
print()

# ---------- Durability is dominated by MTTR, not by disk quality ----------
print("Sensitivity of EC 10+4 to how fast you repair:")
for hrs in (1, 4, 24, 24 * 7):
    p = loss_prob_repair(10, 4, repair_hours=hrs)
    label = f"{hrs}h" if hrs < 24 else f"{hrs//24}d"
    print(f"  repair in {label:>4}: {p:>9.1e}  ({nines(p):>4.1f} nines)")
print()
print("Halving repair time buys you ~1.2 nines for FREE — no extra hardware.")
print("Now compare buying durability with parity instead:")
for m in (3, 4, 5):
    p = loss_prob_repair(10, m)
    print(f"  EC 10+{m} (overhead {(10+m)/10:.1f}x): {nines(p):>4.1f} nines")
print()
print("→ This is why object stores obsess over rebuild throughput and spread")
print("  shards across thousands of disks: a failed disk's data is rebuilt by")
print("  *all* of them in parallel, which is what makes a 1-4 hour MTTR possible.")

### So why does AWS advertise **11** nines and not 16?

Model B says 10+4 with a 4-hour repair is good for ~16 nines. S3's published
figure is 99.999999999% — eleven. The gap is the whole lesson, and "11 nines"
is worth nothing as an interview answer unless you can explain it:

1. **Every model above assumes independence, and failures are not independent.**
   Disks in a rack share a power supply and a top-of-rack switch. Drives from
   one manufacturing batch fail together. A firmware bug can take out an entire
   drive model across the fleet on the same afternoon. Spreading shards across
   racks, power zones and buildings reduces the correlation; nothing removes it.
   One correlated event is worth more than a decade of independent failures.
2. **The leading cause of data loss is not disks.** It is software bugs, bad
   deploys, and operator error — a shard map overwritten, a lifecycle rule with
   the wrong prefix, a "cleanup" job pointed at the wrong bucket. None of that
   appears anywhere in the binomial. Versioning, MFA-delete and immutable
   backups exist because of this line, not because of disk failure.
3. **Silent corruption is a different failure mode.** A disk that returns the
   wrong bytes without erroring is invisible to the model — and if you rebuild
   a shard *from* corrupt data, you propagate the damage. This is what
   end-to-end checksums and background **scrubbing** defend against: they turn
   "the bytes are present" into "the bytes are correct".
4. **It is a design floor, not a measurement.** You cannot observe eleven nines:
   confirming it would take on the order of 10¹¹ object-years. It is a target
   chosen so that it stays true under pessimistic assumptions about points 1–3,
   with the independent-failure math left with plenty of headroom.

And one more distinction people fumble: **durability is not availability.**
S3 targets 11 nines of durability and *four* of availability. An AZ outage means
you cannot read your object today; it does not mean the object is gone. Design
for both separately — the mitigations are completely different (redundancy and
scrubbing vs. failover and retries).

## 🗺️ High-level architecture

```
   [Client]
       │  HTTP + SigV4 / HMAC signature
       ▼
  ┌─────────────────────────────────────────┐
  │ API front-end (stateless, autoscaled)   │  parse, authN/authZ, rate-limit
  └──────────────┬──────────────────────────┘
         │       │                 │
         ▼       ▼                 ▼
     Auth/IAM  Metadata         Data plane
               (key → shard     (stores the actual bytes as
               locations)        erasure-coded shards on many
                                 storage nodes across racks/AZs)
                                         │
                                         ▼
                                Cross-region replication (async)
```

Two ideas carry most of the weight:

1. **Separate metadata from data.** Metadata is tiny but needs fast
   lookups and transactions — perfect for a sharded database.
   Data is huge but only needs sequential reads/writes — perfect for
   cheap commodity disks.
2. **Stateless front-ends**, stateful storage. Autoscale the pane we
   can; plan carefully for the pane we can't.


## ✅ What's next

- **Notebook 2** — a progressively-better toy S3 in Python: naive dict
  → versioned store → multipart upload.
- **Notebook 3** — deep dives: presigned URLs (with tamper tests),
  consistent hashing for shard placement, XOR-parity erasure coding,
  and a lifecycle policy engine.
